# Stage 1: 椎体検出器 (YOLOv8) 訓練

**実行環境**: Google Colab（GPU 必須）

## 手順
1. Google Drive をマウントし、リポジトリと `train/dataset/phase2/` を配置
2. 設定セルを編集して実行
3. 訓練済み weights を `train/runs/detector_best.pt` に保存

## BBoxラベル
ランドマーク座標から自動導出されます（別途アノテーション不要）。

In [ ]:
# Google Drive マウント
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys, os
from pathlib import Path

# ---- 設定 (ここを編集) ----
REPO_ROOT   = Path("/content/drive/MyDrive/anglist")  # リポジトリのパス
DATA_DIR    = REPO_ROOT / "train/dataset/phase2"
RUNS_DIR    = REPO_ROOT / "train/runs"
YOLO_MODEL  = "yolov8s.pt"    # yolov8n / yolov8s / yolov8m
EPOCHS      = 100
BATCH_SIZE  = 16
IMG_SIZE    = 640
# ---------------------------

sys.path.insert(0, str(REPO_ROOT))
os.makedirs(str(RUNS_DIR), exist_ok=True)
print("DATA_DIR:", DATA_DIR)

In [ ]:
# 依存インストール
!pip install ultralytics PyYAML -q

In [ ]:
# YOLO形式ラベルとデータセット設定ファイルを生成
import yaml, shutil
import numpy as np
from train.dataset_phase2 import Phase2Dataset, write_yolo_labels
from train.landmark_scheme import BBOX_CLASSES

YOLO_DATASET_DIR = Path("/content/yolo_dataset")
LABEL_DIR = YOLO_DATASET_DIR / "labels/train"
IMAGE_DIR = YOLO_DATASET_DIR / "images/train"
LABEL_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

# BBoxラベル生成
ds = Phase2Dataset(str(DATA_DIR), mode="stage1", resize=(IMG_SIZE, IMG_SIZE))
write_yolo_labels(ds, str(LABEL_DIR))
print(f"Labels: {len(list(LABEL_DIR.glob('*.txt')))} files")

# 画像を PNG として書き出し
import cv2, torch
for i in range(len(ds)):
    item = ds[i]
    if len(item['labels']) == 0:
        continue
    img = (item['image'].squeeze().numpy() * 255).astype(np.uint8)
    cv2.imwrite(str(IMAGE_DIR / f"{item['case_id']}.png"), img)
print(f"Images: {len(list(IMAGE_DIR.glob('*.png')))} files")

# data.yaml
data_yaml = {
    "path": str(YOLO_DATASET_DIR),
    "train": "images/train",
    "val":   "images/train",  # 小規模データのためtrain=valで動作確認
    "nc":    len(BBOX_CLASSES),
    "names": BBOX_CLASSES,
}
yaml_path = YOLO_DATASET_DIR / "data.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(data_yaml, f, allow_unicode=True)
print("data.yaml:", yaml_path)

In [ ]:
# YOLOv8 訓練
from ultralytics import YOLO

model = YOLO(YOLO_MODEL)
results = model.train(
    data=str(yaml_path),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    project=str(RUNS_DIR),
    name="detector",
    exist_ok=True,
)

In [ ]:
# ONNX エクスポート
best_pt = RUNS_DIR / "detector/weights/best.pt"
best_model = YOLO(str(best_pt))
best_model.export(format="onnx", imgsz=IMG_SIZE, opset=17)
print("ONNX exported:", str(best_pt).replace('.pt', '.onnx'))